In [ ]:
import random
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

class manbot():
    def __init__(self, name, coop_stat):
        self.name = name


        self.coop_stat = coop_stat

    def __repr__(self):
        return f"manbot(name={self.name}, coop_stat={self.coop_stat})"


class node():
    def __init__(self, xpos, ypos):
        self.xpos = xpos
        self.ypos = ypos

    def __repr__(self):
        return f"node(xpos={self.xpos:.2f}, ypos={self.ypos:.2f})"


class mapper():
    def __init__(self, repete=100, grid=20):
        self.grid = grid
        self.node_dic = {}
        self.layout_choice = 1

        print("(1). Random Network (2). Square Grid (3). Linear Network (4). Double Spanning Tree (5). ODE Single Node")

        self.layout_choice = int(input("Layout Choice: "))

        if self.layout_choice in [1,4]:
            i = 0
            while i < repete: 
                xx = random.uniform(0, grid)
                yy = random.uniform(0, grid)
    
                new = node(xx, yy)
                # Ensure unique node coordinates
                if not any(abs(n.xpos - xx) < 1e-5 and abs(n.ypos - yy) < 1e-5 for n in self.node_dic):
                    self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                i += 1

        if self.layout_choice == 2:
            i = 0
            while i <= grid:
                y = 0
                while y <= grid:
                    new = node(i,y)
                    self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                    y += 1
                i += 1

        if self.layout_choice == 3:
            i = 0
            while i <= repete:

                x_val = random.uniform(0, grid)
                new = node(i, 10)
                if new not in self.node_dic.keys():
                    self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                i += 1

        if self.layout_choice == 5:
            xx = random.uniform(0, grid)
            yy = random.uniform(0, grid)

            new = node(xx,yy)
            self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
            
                
class connector():
    def __init__(self, node_dic, connection_choice, av=25.0):
        self.nodes = node_dic
        # Coerce in case input() handed us a string like "2"
        self.connection_choice = int(connection_choice)
        nodes_list = list(self.nodes.keys())


        # 1. Create connections based on a threshold radius
        if self.connection_choice in [1, 4]:
            for node1 in self.nodes:
                for node2 in self.nodes:
                    if node1 != node2:
                        dist = (((node1.xpos - node2.xpos) ** 2) +
                                ((node1.ypos - node2.ypos) ** 2)) ** 0.5

                        if dist <= av:
                            if node2 not in self.nodes[node1]['connections']:
                                self.nodes[node1]['connections'].append(node2)
                                self.nodes[node2]['connections'].append(node1)

            # 2. Guarantee no node remains completely isolated
            for node1 in self.nodes:
                if len(self.nodes[node1]['connections']) == 0:
                    closest_node = None
                    closest_dist = float('inf')

                    for node2 in self.nodes:
                        if node1 != node2:
                            dist = (((node1.xpos - node2.xpos) ** 2) +
                                    ((node1.ypos - node2.ypos) ** 2)) ** 0.5

                            if dist < closest_dist:
                                closest_dist = dist
                                closest_node = node2

                    if closest_node:
                        self.nodes[node1]['connections'].append(closest_node)

                        if node1 not in self.nodes[closest_node]['connections']:
                            self.nodes[closest_node]['connections'].append(node1)

            # 3. Build a graph representation to inspect overall component connection
            G = nx.Graph()
            for n in self.nodes:
                G.add_node(n)
            for n in self.nodes:
                for neighbor in self.nodes[n]['connections']:
                    G.add_edge(n, neighbor)

            # 4. Bridge sub-graphs safely using a minimum-spanning approach
            while not nx.is_connected(G):
                components = list(nx.connected_components(G))
                if len(components) <= 1:
                    break

                best_pair = None
                best_dist = float('inf')

                for i in range(len(components)):
                    for j in range(i + 1, len(components)):
                        comp1 = components[i]
                        comp2 = components[j]

                        for n1 in comp1:
                            for n2 in comp2:
                                dist = (((n1.xpos - n2.xpos) ** 2) +
                                        ((n1.ypos - n2.ypos) ** 2)) ** 0.5

                                if dist < best_dist:
                                    best_dist = dist
                                    best_pair = (n1, n2)

                if best_pair:
                    node1, node2 = best_pair
                    self.nodes[node1]['connections'].append(node2)
                    self.nodes[node2]['connections'].append(node1)
                    G.add_edge(node1, node2)

        elif self.connection_choice == 2:

            # Build a coord -> node lookup so we can find existing grid neighbors.
            # Without this, `node(x, y) in self.nodes.keys()` was always False
            # because a freshly-constructed node has a different identity.
            by_coord = {(n.xpos, n.ypos): n for n in self.nodes}

            for n in self.nodes:
                left  = by_coord.get((n.xpos - 1, n.ypos))
                right = by_coord.get((n.xpos + 1, n.ypos))
                down  = by_coord.get((n.xpos, n.ypos - 1))
                up    = by_coord.get((n.xpos, n.ypos + 1))

                if left is not None:
                    self.nodes[n]['connections'].append(left)
                if right is not None:
                    self.nodes[n]['connections'].append(right)
                if down is not None:
                    self.nodes[n]['connections'].append(down)
                if up is not None:
                    self.nodes[n]['connections'].append(up)

            G = nx.Graph()
            for n in self.nodes:
                G.add_node(n)
            for n in self.nodes:
                for neighbor in self.nodes[n]['connections']:
                    G.add_edge(n, neighbor)

        elif self.connection_choice == 3:

            for n in self.nodes:
                pluslist = []
                negativelist = []
                for n2 in self.nodes:

                    temp_dist = n.xpos - n2.xpos

                    if temp_dist > 0:

                        pluslist.append([temp_dist, n2])

                    if temp_dist < 0:

                        negativelist.append([abs(temp_dist), n2])


                # Guard against empty lists (end-of-line nodes have no neighbor on one side)
                if pluslist:
                    leftside = min(pluslist)
                    self.nodes[n]['connections'].append(leftside[1])
                if negativelist:
                    rightside = min(negativelist)
                    self.nodes[n]['connections'].append(rightside[1])

            G = nx.Graph()
            for n in self.nodes:
                G.add_node(n)
            for n in self.nodes:
                for neighbor in self.nodes[n]['connections']:
                    G.add_edge(n, neighbor)

        elif self.connection_choice == 4:

            # Clear any existing connections
            for n in nodes_list:
                self.nodes[n]['connections'] = []

            TARGET_EDGES = 760  # 20x20 grid graph: 2 * 20 * 19

            # =============================================================
            # SPANNING TREE 1 - build into temp dict tree1_conns
            # =============================================================
            tree1_conns = {n: [] for n in nodes_list}

            # Shuffle iteration order for variety
            rng1 = random.Random(1)
            order1 = list(nodes_list)
            rng1.shuffle(order1)

            # Phase 2 of tree 1: nearest-neighbor pairing
            for node1 in order1:
                if len(tree1_conns[node1]) == 0:
                    closest_node = None
                    closest_dist = float('inf')
                    for node2 in nodes_list:
                        if node1 is node2:
                            continue
                        dist = (((node1.xpos - node2.xpos) ** 2) +
                                ((node1.ypos - node2.ypos) ** 2)) ** 0.5
                        if dist < closest_dist:
                            closest_dist = dist
                            closest_node = node2
                    if closest_node is not None:
                        tree1_conns[node1].append(closest_node)
                        if node1 not in tree1_conns[closest_node]:
                            tree1_conns[closest_node].append(node1)

            # Phase 4 of tree 1: bridge components
            G1 = nx.Graph()
            for n in nodes_list:
                G1.add_node(n)
            for n in nodes_list:
                for nb in tree1_conns[n]:
                    G1.add_edge(n, nb)

            while not nx.is_connected(G1):
                components = list(nx.connected_components(G1))
                if len(components) <= 1:
                    break
                best_pair = None
                best_dist = float('inf')
                for i in range(len(components)):
                    for j in range(i + 1, len(components)):
                        for n1 in components[i]:
                            for n2 in components[j]:
                                dist = (((n1.xpos - n2.xpos) ** 2) +
                                        ((n1.ypos - n2.ypos) ** 2)) ** 0.5
                                if dist < best_dist:
                                    best_dist = dist
                                    best_pair = (n1, n2)
                if best_pair:
                    a, b = best_pair
                    tree1_conns[a].append(b)
                    tree1_conns[b].append(a)
                    G1.add_edge(a, b)

            # Collect tree 1 edges into a set of frozensets (for forbidding in tree 2)
            tree1_edge_set = set()
            for n in nodes_list:
                for nb in tree1_conns[n]:
                    tree1_edge_set.add(frozenset({id(n), id(nb)}))

            # =============================================================
            # SPANNING TREE 2 - same logic but tree 1's edges are forbidden
            # =============================================================
            tree2_conns = {n: [] for n in nodes_list}

            rng2 = random.Random(2)
            order2 = list(nodes_list)
            rng2.shuffle(order2)

            # Phase 2 of tree 2: nearest-neighbor pairing, skipping tree 1 edges
            for node1 in order2:
                if len(tree2_conns[node1]) == 0:
                    closest_node = None
                    closest_dist = float('inf')
                    for node2 in nodes_list:
                        if node1 is node2:
                            continue
                        if frozenset({id(node1), id(node2)}) in tree1_edge_set:
                            continue
                        dist = (((node1.xpos - node2.xpos) ** 2) +
                                ((node1.ypos - node2.ypos) ** 2)) ** 0.5
                        if dist < closest_dist:
                            closest_dist = dist
                            closest_node = node2
                    if closest_node is not None:
                        tree2_conns[node1].append(closest_node)
                        if node1 not in tree2_conns[closest_node]:
                            tree2_conns[closest_node].append(node1)

            # Phase 4 of tree 2: bridge components, skipping tree 1 edges where possible
            G2 = nx.Graph()
            for n in nodes_list:
                G2.add_node(n)
            for n in nodes_list:
                for nb in tree2_conns[n]:
                    G2.add_edge(n, nb)

            while not nx.is_connected(G2):
                components = list(nx.connected_components(G2))
                if len(components) <= 1:
                    break
                best_pair = None
                best_dist = float('inf')
                for i in range(len(components)):
                    for j in range(i + 1, len(components)):
                        for n1 in components[i]:
                            for n2 in components[j]:
                                if frozenset({id(n1), id(n2)}) in tree1_edge_set:
                                    continue
                                dist = (((n1.xpos - n2.xpos) ** 2) +
                                        ((n1.ypos - n2.ypos) ** 2)) ** 0.5
                                if dist < best_dist:
                                    best_dist = dist
                                    best_pair = (n1, n2)
                # Last resort: if no non-forbidden bridge available, allow forbidden
                if best_pair is None:
                    for i in range(len(components)):
                        for j in range(i + 1, len(components)):
                            for n1 in components[i]:
                                for n2 in components[j]:
                                    dist = (((n1.xpos - n2.xpos) ** 2) +
                                            ((n1.ypos - n2.ypos) ** 2)) ** 0.5
                                    if dist < best_dist:
                                        best_dist = dist
                                        best_pair = (n1, n2)
                if best_pair:
                    a, b = best_pair
                    tree2_conns[a].append(b)
                    tree2_conns[b].append(a)
                    G2.add_edge(a, b)

            # =============================================================
            # MERGE both trees into self.nodes[...]['connections']
            # =============================================================
            applied = set()
            for n in nodes_list:
                for nb in tree1_conns[n]:
                    key = frozenset({id(n), id(nb)})
                    if key in applied:
                        continue
                    applied.add(key)
                    if nb not in self.nodes[n]['connections']:
                        self.nodes[n]['connections'].append(nb)
                    if n not in self.nodes[nb]['connections']:
                        self.nodes[nb]['connections'].append(n)
            for n in nodes_list:
                for nb in tree2_conns[n]:
                    key = frozenset({id(n), id(nb)})
                    if key in applied:
                        continue
                    applied.add(key)
                    if nb not in self.nodes[n]['connections']:
                        self.nodes[n]['connections'].append(nb)
                    if n not in self.nodes[nb]['connections']:
                        self.nodes[nb]['connections'].append(n)

            current = len(applied)

            # =============================================================
            # ADJUST EDGE COUNT TO TARGET
            # =============================================================
            if current < TARGET_EDGES:
                # Densify: add shortest unconnected pairs until we hit target
                candidates = []
                for i in range(len(nodes_list)):
                    n1 = nodes_list[i]
                    for j in range(i + 1, len(nodes_list)):
                        n2 = nodes_list[j]
                        if frozenset({id(n1), id(n2)}) in applied:
                            continue
                        dist = (((n1.xpos - n2.xpos) ** 2) +
                                ((n1.ypos - n2.ypos) ** 2)) ** 0.5
                        candidates.append((dist, n1, n2))
                candidates.sort(key=lambda x: x[0])
                needed = TARGET_EDGES - current
                for dist, n1, n2 in candidates[:needed]:
                    self.nodes[n1]['connections'].append(n2)
                    self.nodes[n2]['connections'].append(n1)
                    applied.add(frozenset({id(n1), id(n2)}))

            elif current > TARGET_EDGES:
                # Trim: remove longest edges that aren't bridges (won't disconnect graph)
                G = nx.Graph()
                for n in nodes_list:
                    G.add_node(n)
                for n in nodes_list:
                    for nb in self.nodes[n]['connections']:
                        G.add_edge(n, nb)

                seen = set()
                edges_with_dist = []
                for n in nodes_list:
                    for nb in self.nodes[n]['connections']:
                        key = frozenset({id(n), id(nb)})
                        if key in seen:
                            continue
                        seen.add(key)
                        dist = (((n.xpos - nb.xpos) ** 2) +
                                ((n.ypos - nb.ypos) ** 2)) ** 0.5
                        edges_with_dist.append((dist, n, nb))
                edges_with_dist.sort(key=lambda x: -x[0])  # longest first

                to_remove = current - TARGET_EDGES
                bridges = set(frozenset({id(a), id(b)}) for a, b in nx.bridges(G))
                removed = 0
                for dist, a, b in edges_with_dist:
                    if removed >= to_remove:
                        break
                    key = frozenset({id(a), id(b)})
                    if key in bridges:
                        continue
                    self.nodes[a]['connections'].remove(b)
                    self.nodes[b]['connections'].remove(a)
                    G.remove_edge(a, b)
                    bridges = set(frozenset({id(x), id(y)}) for x, y in nx.bridges(G))
                    removed += 1



        elif self.connection_choice == 5:

            for nod in self.nodes:
                self.nodes[nod]['connections'].append(nod)
            G = nx.Graph()
            for n in self.nodes:
                G.add_node(n)
            for n in self.nodes:
                for neighbor in self.nodes[n]['connections']:
                    G.add_edge(n, neighbor)
    
        

In [ ]:
import copy

class populator():
    def __init__(self, nodee_dic, name_count=1):
        self.nodd = nodee_dic
        self.name_count = name_count
        chance = [1, 2, 3]

        for nodd in self.nodd:
            self.nodd[nodd]['inhabitants'].clear()
                
                

        for nodd in self.nodd:
            i = 0 
            while i < 5:
                bot_name = 'Bot' + str(self.name_count)
                coop_stat = random.choice(chance)
                new_bot = manbot(bot_name, coop_stat)
    
                nodee_dic[nodd]['inhabitants'].append(new_bot)
                self.name_count += 1
                i += 1


class mover():
    def __init__(self, node_dicc, node, bot):
        self.node_dic = node_dicc
        self.node = node
        self.bot = bot
    
        chancer_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        chance = random.choice(chancer_list)
        chance2 = random.choice(chancer_list)

        # Random death: ~5% chance per step (chance == 10 AND chance2 in [1,2,3,4,5])
        if chance == 10 and chance2 in [1, 2, 3, 4, 5]:
            if self.bot in self.node_dic[self.node]['inhabitants']:
                self.node_dic[self.node]['inhabitants'].remove(self.bot)

        else: #if not randomly killed...

            destination = random.choice(self.node_dic[self.node]['connections'])
    
            # carrying capacity - cap lowered from 10 to 7
            if (len(self.node_dic[self.node]['inhabitants']) > 10):
    
                over = (len(self.node_dic[self.node]['inhabitants']) - 10)/10
    
                chance = random.random() 
    
                # When overcrowded eviction fires, 50% chance the bot dies
                # instead of relocating. Previously bot always moved.
                if chance <= over:
                    if random.random() <= 0.5:
                        # Death from overcrowding
                        if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
                    else:
                        # Relocation with a chance of death if destination is too crowded
                        self.node_dic[destination]['inhabitants'].append(self.bot)
                        
                        if self.bot in self.node_dic[self.node]['inhabitants']:
                            
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
                            
                        if len(self.node_dic[destination]['inhabitants']) > 20:

                            over = (len(self.node_dic[self.node]['inhabitants']) - 20)/20
    
                            chance = random.random() 
    
                            if (chance*2) <= over:
                                self.node_dic[destination]['inhabitants'].remove(self.bot)
                            
                            
        
            #if no other inhabitants in node, 40% chance they move
            elif (len(self.node_dic[self.node]['inhabitants']) < 2):
                if 4 >= (10*random.random()):
                        
                    self.node_dic[destination]['inhabitants'].append(self.bot)
                    if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)

            #random chance of moving, accompanied by 20% chance of death
            elif 1 >= (10*random.random()):
                        
                    self.node_dic[destination]['inhabitants'].append(self.bot)
                    if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
    
                    if  2 >= (10*random.random()):
                        if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
    
       
            


class player():
    def __init__(self, node_dicc, node, name_count):
        self.node_dic = node_dicc
        self.node = node
        self.name_count = name_count
    
        deaths = []
        births = []

        fight_club = list(self.node_dic[node]['inhabitants'])
        

        # Only play a game if there are at least two bots on the node
        for bot1 in fight_club:

            if len(fight_club) > 1:
                
                
                bot2 = random.choice(fight_club)
    
                # SCENARIO 1: Both Cooperate (Coop vs Coop)
                # 60% chance of reproducing a new cooperator
                if (bot1.coop_stat in [1,2]) and (bot2.coop_stat in [1,2]):
                    if (random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4,5,6]):
                        self.name_count += 1
                        new_bot = manbot('Bot' + str(self.name_count), bot1.coop_stat)
                        births.append(new_bot)
            
                # SCENARIO 2: Bot1 Cooperates, Bot2 Defects (Coop vs Defect)
                # The cooperator (bot1) dies, 40%
                elif (bot1.coop_stat in [1,2]) and (bot2.coop_stat == 3):
                    
                    if (random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4]):

                        if bot1 not in deaths:
                            deaths.append(bot1)
                        
                # SCENARIO 3: Bot1 Defects, Bot2 Cooperates (Defect vs Coop)
                # The defector (bot1) reproduces another defector, 80%
                elif (bot1.coop_stat == 3) and (bot2.coop_stat in [1,2]):
                    
                    if (random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4,5,6,7,8]):
                        self.name_count += 1
                        new_bot = manbot('Bot' + str(self.name_count), bot1.coop_stat)
                        births.append(new_bot)
            
                # SCENARIO 4: Both Defect (Defect vs Defect)
                # Mutual destruction: both defectors die 20%
                elif (bot1.coop_stat == 3) and (bot2.coop_stat == 3):
                    if (bot1 not in deaths):
                        if (random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2]):
                            deaths.append(bot1)
                    elif (bot2 not in deaths):
                        if (random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2]):
                            deaths.append(bot2)


            

        for dead in deaths:
            if dead in self.node_dic[node]['inhabitants']:
                self.node_dic[node]['inhabitants'].remove(dead)

        self.node_dic[node]['inhabitants'].extend(births)
        births.clear()
                    



def execute_simulation_step(node_dic, current_count):
    """Executes a single step of spatial shifting and player interactions across the dictionary."""
    for nodel in node_dic:
        inhabitants = list(node_dic[nodel]['inhabitants'])
        if len(inhabitants) > 0:
            # Shift bots based on movement rules

            for bott in node_dic[nodel]['inhabitants']:
                my_mover = mover(node_dic, nodel, bott)

            # Handle game outcomes
    for nodel in node_dic:
        p = player(node_dic, nodel, current_count)
        current_count = p.name_count

    return current_count

In [ ]:
my_mapper = mapper(repete=50, grid=20)
my_connector = connector(my_mapper.node_dic, my_mapper.layout_choice, av=1.0)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
def animate_evolving_population_with_strategy(nodess, n_iterations=50, gridsize = 20, frame_count = 10):

 
    my_populator = populator(nodess)

    #print (my_populator.nodd)
    
    node_dic = nodess
    current_count = my_populator.name_count
    
    # Create figure with extra room on the right for a colorbar legend
    fig, ax = plt.subplots(figsize=(15, 7))
    
    # Pre-build NetworkX layout
    G = nx.Graph()
    pos = {}
    for node_obj in node_dic:
        G.add_node(node_obj)
        pos[node_obj] = (node_obj.xpos, node_obj.ypos)
        for target in node_dic[node_obj]['connections']:
            G.add_edge(node_obj, target)


    #Fix colorbar (added purple)
    custom_colors = ['#d62728', '#9467bd', '#1f77b4']
    rpb_cmap = LinearSegmentedColormap.from_list("RedPurpleBlue", custom_colors)
    
    # Create a persistent colorbar axis so it doesn't duplicate every frame
    cax = fig.add_axes([0.88, 0.15, 0.03, 0.7])
    sm = plt.cm.ScalarMappable(cmap=rpb_cmap, norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Strategy Dominance (Defection vs Cooperation)', rotation=270, labelpad=15, fontweight='bold')
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Defection', '50 / 50 Mix', 'Cooperation'])

    def update(frame):
        nonlocal current_count
        ax.clear()
        
        if frame > 0:
            i = 0
            while i < frame_count:
                current_count = execute_simulation_step(node_dic, current_count)
                i += 1
            
        # Lists to separate populated vs empty nodes for distinct styling
        live_nodes = []
        live_sizes = []
        live_colors = []
        
        empty_nodes = []
        
        total_bots = 0
        
        # Categorize nodes based on active population
        for node_obj in G.nodes:
            bots = node_dic[node_obj]['inhabitants']
            population_size = len(bots)
            total_bots += population_size
            
            if population_size > 0:
                live_nodes.append(node_obj)
                live_sizes.append(50 + population_size * 12)
                
                # Strategy ratio tracking
                cooperators = sum(1 for b in bots if b.coop_stat in [1, 2])
                coop_ratio = cooperators / population_size
                live_colors.append(coop_ratio)
            else:
                empty_nodes.append(node_obj)
                
        # 1. Draw standard background connection edges
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gainsboro', width=1.2)
        
        # 2. Draw live nodes with filled strategy colors (Red to Blue)
        if live_nodes:
            nx.draw_networkx_nodes(
                G, pos, ax=ax, nodelist=live_nodes, node_size=live_sizes, 
                node_color=live_colors, cmap=plt.cm.RdBu, vmin=0, vmax=1
            )
            
        # 3. Draw empty nodes as completely hollow, thin outlines
        if empty_nodes:
            nx.draw_networkx_nodes(
                G, pos, ax=ax, nodelist=empty_nodes, 
                node_size=40,            # Small uniform baseline size
                node_color='none',        # Transparent/hollow center
                edgecolors='dimgray',     # Clear gray border ring
                linewidths=1.5,           # Noticeable border thickness
                node_shape='o'            # Standard circle shape
            )
            
            # Optional: To make the hollow circles dashed, we grab the 
            # last drawn collection on the axis and apply a dash style
            ax.collections[-1].set_linestyle((0, (3, 3)))
        
        # Consistent layout limits
        all_x = [n.xpos for n in G.nodes]
        all_y = [n.ypos for n in G.nodes]
        x_min, x_max = min(all_x), max(all_x)
        y_min, y_max = min(all_y), max(all_y)
        x_pad = max((x_max - x_min) * 0.05, 0.5)
        y_pad = max((y_max - y_min) * 0.05, 0.5)
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)

    ani = FuncAnimation(fig, update, frames=(n_iterations//frame_count), repeat=False)
    plt.close(fig)
    return ani

In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 125.0



strategy_animation = animate_evolving_population_with_strategy(my_mapper.node_dic, n_iterations=3500, gridsize = 20)
HTML(strategy_animation.to_jshtml())



In [ ]:
strategy_animation.save("network_evolution4.gif", writer = "pillow", fps = 5)